In [ ]:
# --- 路径修复：notebook 在 Base/ 下，数据在 Data/ 下 ---
import os as _os, sys as _sys
if _os.path.basename(_os.getcwd()) == 'Base':
    _os.chdir('..')
if 'Base' not in _sys.path:
    _sys.path.insert(0, 'Base')

def _find(path):
    """自动检测 Data/ 子目录前缀。"""
    if _os.path.exists(path):
        return path
    alt = _os.path.join('Data', path)
    if _os.path.exists(alt):
        return alt
    return path


# CNstar data acquisition

In [7]:
import numpy as np
import math
import fits
import astropy.io.fits as afits
import matplotlib.pyplot as plt

In [8]:
import os
import requests
import time
import re
from concurrent.futures import ThreadPoolExecutor, as_completed


In [3]:
def download_single_file(url, output_dir, file_index, total_files, retry_count=3):
    """下载单个文件（线程安全的函数）"""
    # 检查文件是否已存在（先临时用URL解析的文件名做预判，后续会替换）
    temp_file_id = url.split('/')[-1].split('?')[0]
    temp_filepath = os.path.join(output_dir, temp_file_id)
    if os.path.exists(temp_filepath):
        return True, temp_file_id, "已存在"

    # 尝试下载
    for attempt in range(retry_count):
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()

            # 从响应头获取原始文件名（核心修复）
            filename = None
            if 'Content-Disposition' in response.headers:
                content_disposition = response.headers['Content-Disposition']
                # 解析 filename*=UTF-8''xxx 或 filename="xxx" 格式
                match = re.search(r'filename\*?=([^;]+)', content_disposition)
                if match:
                    filename_encoded = match.group(1).strip()
                    if filename_encoded.startswith("UTF-8''"):
                        filename = filename_encoded[7:]  # 处理UTF-8编码的文件名
                    else:
                        filename = filename_encoded.strip('"')  # 处理普通带引号的文件名

            # 如果响应头没有文件名， fallback 到URL解析
            if not filename:
                filename = url.split('/')[-1].split('?')[0]

            filepath = os.path.join(output_dir, filename)

            # 保存文件
            with open(filepath, 'wb') as f:
                f.write(response.content)

            file_size = len(response.content) / 1024  # KB
            return True, filename, f"成功 ({file_size:.1f} KB)"

        except Exception as e:
            if attempt < retry_count - 1:
                time.sleep(1)
            else:
                return False, temp_file_id, f"失败: {str(e)[:50]}"

    return False, temp_file_id, "重试次数用尽"

def multi_thread_download():
    with open('20260415192129.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()

    output_dir = "dr13_new"
    os.makedirs(output_dir, exist_ok=True)

    urls = []
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('wget'):
            continue
        if line.startswith('http'):
            urls.append(line)

    total = len(urls)
    print(f"找到 {total} 个下载链接")
    print(f"开始多线程下载 (最大线程数: 5)")

    success_count = 0
    failed_count = 0
    skipped_count = 0
    downloaded_files = []

    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_url = {}
        for i, url in enumerate(urls, 1):
            future = executor.submit(download_single_file, url, output_dir, i, total)
            future_to_url[future] = (url, i)

        completed = 0
        for future in as_completed(future_to_url):
            url, index = future_to_url[future]
            completed += 1

            try:
                success, filename, message = future.result()

                if message == "已存在":
                    skipped_count += 1
                    status = "跳过"
                elif success:
                    success_count += 1
                    downloaded_files.append(filename)
                    status = "成功"
                else:
                    failed_count += 1
                    status = "失败"

                progress = (completed / total) * 100
                print(f"[{completed}/{total}] {status}: {filename} - {message}")
                print(f"总进度: {progress:.1f}%")

                if not success and message != "已存在":
                    with open(os.path.join(output_dir, 'failed.txt'), 'a') as f:
                        f.write(f"{url}\n")

            except Exception as e:
                failed_count += 1
                print(f"[{index}/{total}] 异常: {str(e)[:50]}")
                with open(os.path.join(output_dir, 'failed.txt'), 'a') as f:
                    f.write(f"{url}\n")

    print("\n" + "="*50)
    print(f"下载完成!")
    print(f"总计: {total}")
    print(f"成功: {success_count}")
    print(f"失败: {failed_count}")
    print(f"跳过: {skipped_count}")

    if failed_count > 0:
        print(f"失败的URL已保存到: {os.path.join(output_dir, 'failed.txt')}")

    with open(os.path.join(output_dir, 'downloaded_files.txt'), 'w') as f:
        for file in downloaded_files:
            f.write(f"{file}\n")

if __name__ == "__main__":
    start_time = time.time()
    multi_thread_download()
    end_time = time.time()
    total_time = end_time - start_time
    minutes = int(total_time // 60)
    seconds = int(total_time % 60)
    print(f"总耗时: {minutes}分{seconds}秒")

找到 40722 个下载链接
开始多线程下载 (最大线程数: 5)
[1/40722] 成功: spec-55860-B6001_sp03-215.fits.gz - 成功 (56.7 KB)
总进度: 0.0%
[2/40722] 成功: spec-55859-F5907_sp14-096.fits.gz - 成功 (56.6 KB)
总进度: 0.0%
[3/40722] 成功: spec-55860-B6001_sp14-173.fits.gz - 成功 (56.8 KB)
总进度: 0.0%
[4/40722] 成功: spec-55859-F5902_sp10-123.fits.gz - 成功 (56.8 KB)
总进度: 0.0%
[5/40722] 成功: spec-55859-F5907_sp03-160.fits.gz - 成功 (56.9 KB)
总进度: 0.0%
[6/40722] 成功: spec-55862-B6202_sp09-127.fits.gz - 成功 (56.9 KB)
总进度: 0.0%
[7/40722] 成功: spec-55862-B6212_sp08-161.fits.gz - 成功 (56.8 KB)
总进度: 0.0%
[8/40722] 成功: spec-55862-B6202_sp10-166.fits.gz - 成功 (56.7 KB)
总进度: 0.0%
[9/40722] 成功: spec-55862-B6212_sp10-004.fits.gz - 成功 (56.7 KB)
总进度: 0.0%
[10/40722] 成功: spec-55862-B6202_sp01-159.fits.gz - 成功 (57.3 KB)
总进度: 0.0%
[11/40722] 成功: spec-55863-B6301_sp03-121.fits.gz - 成功 (56.8 KB)
总进度: 0.0%
[12/40722] 成功: spec-55862-B6202_sp16-231.fits.gz - 成功 (56.7 KB)
总进度: 0.0%
[13/40722] 成功: spec-55863-B6302_sp02-101.fits.gz - 成功 (56.7 KB)
总进度: 0.0%
[14/40722] 成功

In [6]:
def download_single_file(url, output_dir, file_index, total_files, retry_count=3):
    """下载单个文件（线程安全的函数）"""
    # 检查文件是否已存在（先临时用URL解析的文件名做预判，后续会替换）
    temp_file_id = url.split('/')[-1].split('?')[0]
    temp_filepath = os.path.join(output_dir, temp_file_id)
    if os.path.exists(temp_filepath):
        return True, temp_file_id, "已存在"

    # 尝试下载
    for attempt in range(retry_count):
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()

            # 从响应头获取原始文件名（核心修复）
            filename = None
            if 'Content-Disposition' in response.headers:
                content_disposition = response.headers['Content-Disposition']
                # 解析 filename*=UTF-8''xxx 或 filename="xxx" 格式
                match = re.search(r'filename\*?=([^;]+)', content_disposition)
                if match:
                    filename_encoded = match.group(1).strip()
                    if filename_encoded.startswith("UTF-8''"):
                        filename = filename_encoded[7:]  # 处理UTF-8编码的文件名
                    else:
                        filename = filename_encoded.strip('"')  # 处理普通带引号的文件名

            # 如果响应头没有文件名， fallback 到URL解析
            if not filename:
                filename = url.split('/')[-1].split('?')[0]

            filepath = os.path.join(output_dir, filename)

            # 保存文件
            with open(filepath, 'wb') as f:
                f.write(response.content)

            file_size = len(response.content) / 1024  # KB
            return True, filename, f"成功 ({file_size:.1f} KB)"

        except Exception as e:
            if attempt < retry_count - 1:
                time.sleep(1)
            else:
                return False, temp_file_id, f"失败: {str(e)[:50]}"

    return False, temp_file_id, "重试次数用尽"

def multi_thread_download():
    with open('20260415225520.txt', 'r', encoding='utf-8') as f:
        lines = f.readlines()

    output_dir = "af_0"
    os.makedirs(output_dir, exist_ok=True)

    urls = []
    for line in lines:
        line = line.strip()
        if not line or line.startswith('#') or line.startswith('wget'):
            continue
        if line.startswith('http'):
            urls.append(line)

    total = len(urls)
    print(f"找到 {total} 个下载链接")
    print(f"开始多线程下载 (最大线程数: 5)")

    success_count = 0
    failed_count = 0
    skipped_count = 0
    downloaded_files = []

    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_url = {}
        for i, url in enumerate(urls, 1):
            future = executor.submit(download_single_file, url, output_dir, i, total)
            future_to_url[future] = (url, i)

        completed = 0
        for future in as_completed(future_to_url):
            url, index = future_to_url[future]
            completed += 1

            try:
                success, filename, message = future.result()

                if message == "已存在":
                    skipped_count += 1
                    status = "跳过"
                elif success:
                    success_count += 1
                    downloaded_files.append(filename)
                    status = "成功"
                else:
                    failed_count += 1
                    status = "失败"

                progress = (completed / total) * 100
                print(f"[{completed}/{total}] {status}: {filename} - {message}")
                print(f"总进度: {progress:.1f}%")

                if not success and message != "已存在":
                    with open(os.path.join(output_dir, 'failed.txt'), 'a') as f:
                        f.write(f"{url}\n")

            except Exception as e:
                failed_count += 1
                print(f"[{index}/{total}] 异常: {str(e)[:50]}")
                with open(os.path.join(output_dir, 'failed.txt'), 'a') as f:
                    f.write(f"{url}\n")

    print("\n" + "="*50)
    print(f"下载完成!")
    print(f"总计: {total}")
    print(f"成功: {success_count}")
    print(f"失败: {failed_count}")
    print(f"跳过: {skipped_count}")

    if failed_count > 0:
        print(f"失败的URL已保存到: {os.path.join(output_dir, 'failed.txt')}")

    with open(os.path.join(output_dir, 'downloaded_files.txt'), 'w') as f:
        for file in downloaded_files:
            f.write(f"{file}\n")

if __name__ == "__main__":
    start_time = time.time()
    multi_thread_download()
    end_time = time.time()
    total_time = end_time - start_time
    minutes = int(total_time // 60)
    seconds = int(total_time % 60)
    print(f"总耗时: {minutes}分{seconds}秒")

找到 42 个下载链接
开始多线程下载 (最大线程数: 5)
[1/42] 成功: spec-56394-HD132901N475049B01_sp03-120.fits.gz - 成功 (55.8 KB)
总进度: 2.4%
[2/42] 成功: spec-56376-HD162834N013512B01_sp09-247.fits.gz - 成功 (56.3 KB)
总进度: 4.8%
[3/42] 成功: spec-56423-HD155304N411742B01_sp09-144.fits.gz - 成功 (56.6 KB)
总进度: 7.1%
[4/42] 成功: spec-56254-HD092603S011405M01_sp04-145.fits.gz - 成功 (56.6 KB)
总进度: 9.5%
[5/42] 成功: spec-56424-HD153055N404958B01_sp03-189.fits.gz - 成功 (55.9 KB)
总进度: 11.9%
[6/42] 成功: spec-56618-EG000954N044957V01_sp02-054.fits.gz - 成功 (56.6 KB)
总进度: 14.3%
[7/42] 成功: spec-56675-HD134613N305349V01_sp03-187.fits.gz - 成功 (56.5 KB)
总进度: 16.7%
[8/42] 成功: spec-56667-HD104840S015732B_sp06-122.fits.gz - 成功 (56.6 KB)
总进度: 19.0%
[9/42] 成功: spec-56660-HD083602N240304V01_sp03-241.fits.gz - 成功 (56.5 KB)
总进度: 21.4%
[10/42] 成功: spec-56698-HD151053N291344V_sp11-234.fits.gz - 成功 (55.8 KB)
总进度: 23.8%
[11/42] 成功: spec-56642-HD114322N280318V_sp05-245.fits.gz - 成功 (55.7 KB)
总进度: 26.2%
[12/42] 成功: spec-57083-HD112135N402551V01_sp14-073.fi

In [4]:
star_hdus = afits.open('Data/dr13_v1.0_LRS_stellar.fits.gz') 

In [5]:
star = star_hdus[1].data
star

FITS_rec([(1247701114, 'L14166507495863',          1247701114, 'J231742.68+391654.1', '2024-10-07', 60591, 60590, 'LN231823N414625BM01', 1, 114, 349.4278384, 39.2817017, 15.58,  62.79, 103.08, 112.45,  72.33, 'STAR', 'F9', -9.636670e-05, 2.91869e-05, 155133494274988251, -999.    , -999.    ,   17.9186, -999.    , -999.    , '1917263406548166144', 15.125517, 'Obj', 0, 0., 349.4278384, 39.2817017, 5813.41,  65.56, 4.241, 0.093,  0.288, 0.056,  -28.89,  8.75, -9.999e+03, -9.999e+03, -9999.),
          (1247701115, 'L14166512405841',          1247701115, 'J231655.67+392212.0', '2024-10-07', 60591, 60590, 'LN231823N414625BM01', 1, 115, 349.2319733, 39.3700145,  9.59,  31.71,  46.64,  50.7 ,  29.68, 'STAR', 'F5', -6.497830e-05, 6.54119e-05, 155243492316254353, -999.    , -999.    ,   19.0488, -999.    , -999.    , '1918015987897913216', 16.350792, 'Obj', 0, 0., 349.2319733, 39.3700145, 6329.7 , 169.1 , 4.38 , 0.237, -0.667, 0.167,  -19.48, 19.61,  3.170e-01,  2.500e-02, -9999.),
          (1

In [6]:
print(f"星体总数量: {len(star)}")

星体总数量: 8981588


In [7]:
#  应用筛选条件
mask = (
    (star['teff'] > 4000) & (star['teff'] < 5500) &  # 条件1：有效温度
    (star['logg'] < 3.0) &                           # 条件2：表面重力
    (star['feh'] > -1.8) & (star['feh'] < -1.0) &   # 条件3：金属丰度
    (star['snru'] > 5.0)                            # 条件4：u波段信噪比
)

# 3. 应用筛选条件
selected_stars = star[mask]


In [8]:
print(f"筛选后星体数量: {len(selected_stars)}")

筛选后星体数量: 26687


In [9]:
import pandas as pd
df = pd.DataFrame({
    'RA': selected_stars['ra'],
    'DEC': selected_stars['dec'],
    'Teff': selected_stars['teff'],
    'logg': selected_stars['logg'],
    'FeH': selected_stars['feh'],
    'lmjd':selected_stars['lmjd'],
    'planid':selected_stars['planid'],
    'spid':selected_stars['spid'],
    'fiberid':selected_stars['fiberid'],
    'obsid':selected_stars['obsid'],
    'uid':selected_stars['uid'],
    'rv':selected_stars['rv'],
    'snru':selected_stars['snru'],
    'snrg':selected_stars['snrg'],
    'mag_ps_g':selected_stars['mag_ps_g']
})

# 保存CSV
df.to_csv(_os.path.join('Data', 'stars.csv'), index=False)


d:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [10]:
import pandas as pd

d:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [11]:
def match_coordinates(cat1, cat2, ra_col='RA', dec_col='DEC', tol=1e-4):
    matched = []
    for idx, row in cat1.iterrows():
        ra1, dec1 = row[ra_col], row[dec_col]
        # 计算与 cat2 中所有天体的角度差
        ra_diff = np.abs(cat2[ra_col] - ra1)
        dec_diff = np.abs(cat2[dec_col] - dec1)
        # 同时满足 RA 和 DEC 容差的视为匹配
        match = (ra_diff <= tol) & (dec_diff <= tol)
        matched.append(match.any())
    return np.array(matched)

def main():
    # 读取两个 CSV 文件
    try:
        cn = pd.read_csv(_find('CNstar.csv'))
        stars = pd.read_csv(_find('stars.csv'))
    except FileNotFoundError as e:
        print(f"错误：找不到文件 - {e.filename}")
        return

    # 确保必要的列存在
    required_cols = ['RA', 'DEC']
    for col in required_cols:
        if col not in cn.columns:
            print(f"CNstar.csv 缺少列: {col}")
            return
        if col not in stars.columns:
            print(f"stars.csv 缺少列: {col}")
            return


    # 坐标匹配（容差 0.0001 度，约 0.36 角秒）
    matched = match_coordinates(cn, stars, tol=1e-4)

    # 输出未匹配到的天体
    unmatched = cn[~matched]
    if len(unmatched) == 0:
        print("\n CNstar 中的所有天体都在 stars 中找到了匹配！")
    else:
        print(f"\n共有 {len(unmatched)} 个天体未在 stars 中找到匹配：")
        print(unmatched[['RA', 'DEC']].to_string(index=False))

    # 统计信息
    print(f"\n匹配成功: {matched.sum()} / {len(cn)}")

if __name__ == '__main__':
    main()



共有 18 个天体未在 stars 中找到匹配：
        RA       DEC
272.312586 18.683633
255.591906 13.793537
268.058059 26.255920
 58.221019  7.203154
285.305271 39.972211
245.556824  2.385621
258.215699  8.130489
205.376603 15.485927
268.917218 15.243525
268.834347  5.638831
122.120480  1.946907
239.916920 32.446199
 13.926791 18.720631
150.981537  1.298311
 36.148453 28.035597
210.069275 19.893129
195.094070 -7.636771
286.798645 39.137039

匹配成功: 88 / 106


In [1]:
import pandas as pd

def split_csv_by_flag(input_file):
    """
    根据flag列的值将CSV文件分成两个表格
    
    参数:
    input_file: 输入的CSV文件路径
    """
    # 读取CSV文件
    df = pd.read_csv(input_file)
    
    # 根据flag列的值进行筛选
    df_flag_0 = df[df['flag'] == 0]
    df_flag_1 = df[df['flag'] == 1]
    
    # 将结果保存为新的CSV文件
    df_flag_0.to_csv(_os.path.join('Data', 'output_flag_0.csv'), index=False)
    df_flag_1.to_csv(_os.path.join('Data', 'output_flag_1.csv'), index=False)
    
    print(f"原始表格共有 {len(df)} 行数据")
    print(f"flag为0的表格共有 {len(df_flag_0)} 行数据")
    print(f"flag为1的表格共有 {len(df_flag_1)} 行数据")
    
    return df_flag_0, df_flag_1

# 使用示例
if __name__ == "__main__":
    # 替换为你的CSV文件路径
    input_csv_path = _find('apof_cands.csv')
    
    # 执行分割
    table_0, table_1 = split_csv_by_flag(input_csv_path)

d:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
d:\Anaconda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


原始表格共有 73 行数据
flag为0的表格共有 42 行数据
flag为1的表格共有 31 行数据


## Data labeling

In [ ]:
stars_df = pd.read_csv(_find('stars.csv'))
stars_df.columns = stars_df.columns.str.lower().str.strip()

# 读取原有 CN 星表
cn_df = pd.read_csv(_find('CNstar.csv'))
cn_df.columns = cn_df.columns.str.lower().str.strip()

# 读取新增 CN 星表（请根据实际文件名修改）
cn_df2 = pd.read_csv(_find('FT_cands.csv'))   # <--- 新增数据集文件名
cn_df2.columns = cn_df2.columns.str.lower().str.strip()

# 合并两个 CN 星表，去除重复行（基于 ra, dec）
cn_combined = pd.concat([cn_df, cn_df2], ignore_index=True)
cn_combined = cn_combined.drop_duplicates(subset=['ra', 'dec'])

# 统一数值类型
for col in ['ra', 'dec', 'teff', 'logg', 'feh', 'rv']:
    stars_df[col] = pd.to_numeric(stars_df[col], errors='coerce')

for col in ['lmjd', 'spid', 'fiberid']:
    stars_df[col] = pd.to_numeric(stars_df[col], errors='coerce')

stars_df = stars_df.dropna(subset=['ra', 'dec', 'teff', 'logg', 'feh', 'lmjd', 'uid','spid', 'fiberid','snru','mag_ps_g']).copy()
stars_df = stars_df.reset_index(drop=True)

# 标签：1 = 已知 CN 星，-1 = 未标注
stars_df['label'] = -1

catalog_coords = SkyCoord(
    ra=stars_df['ra'].values * u.deg,
    dec=stars_df['dec'].values * u.deg
)

cn_coords = SkyCoord(
    ra=cn_combined['ra'].values * u.deg,
    dec=cn_combined['dec'].values * u.deg
)

idx, d2d, _ = cn_coords.match_to_catalog_sky(catalog_coords)

match_tolerance = 1.0 * u.arcsec
valid_matches = d2d <= match_tolerance

matched_catalog_idx = idx[valid_matches]
matched_sep_arcsec = d2d[valid_matches].arcsec

# 处理重复匹配，保留最小角距
best_sep = {}
for i_cat, sep in zip(matched_catalog_idx, matched_sep_arcsec):
    if (i_cat not in best_sep) or (sep < best_sep[i_cat]):
        best_sep[i_cat] = sep

matched_unique = np.array(list(best_sep.keys()), dtype=int)
stars_df.loc[matched_unique, 'label'] = 1

print(f"样本总数: {len(stars_df)}")
print(f"已知 CN 星数: {(stars_df['label'] == 1).sum()}")
print(f"未标注样本数: {(stars_df['label'] == -1).sum()}")

样本总数: 48440
已知 CN 星数: 107
未标注样本数: 48333


In [ ]:
stars_df.head()

,ra,dec,teff,logg,feh,lmjd,planid,spid,fiberid,obsid,uid,rv,snru,snrg,mag_ps_g,label
0,348.657761,39.543099,5118.04,2.522,-1.653,60591,LN231823N414625BM01,1,212,1247701212,L14166593776015,-401.52,12.60,45.81,-999.0000,-1
1,350.296975,39.763120,4738.28,1.863,-1.116,60591,LN231823N414625BM01,1,232,1247701232,L14166076095466,-260.75,7.06,43.32,16.2941,-1
2,348.336230,41.897089,4563.05,1.419,-1.233,60591,LN231823N414625BM01,3,99,1247703099,L14170734357722,-224.57,12.88,96.06,15.4476,-1
3,348.209212,42.271151,4951.77,2.366,-0.955,60591,LN231823N414625BM01,3,233,1247703233,L14170773735962,-333.67,10.41,59.74,-999.0000,-1
4,350.188710,41.662744,5228.72,2.547,-1.629,60591,LN231823N414625BM01,4,122,1247704122,L14171102923871,-298.05,29.41,98.30,-999.0000,-1


In [ ]:
#  批量读取光谱 
from tqdm.notebook import tqdm
c = 300000  # km/s
common_wave = np.arange(3800.0, 4500, 1.0)


def read_single_spectrum(row, common_wave, folder):
    #构建光谱文件路径
    lmjd = int(row['lmjd'])
    planid = str(row['planid']).strip()
    spid = int(row['spid'])
    fiberid = int(row['fiberid'])

    filename = os.path.join(folder, f"spec-{lmjd}-{planid}_sp{spid:02d}-{fiberid:03d}.fits.gz")

    if not os.path.exists(filename):
        return None

    try:
        with afits.open(filename, memmap=False) as hdul:
            if len(hdul) < 2 or hdul[1].data is None or len(hdul[1].data) == 0:
                return None

            row_data = hdul[1].data[0]
            wave = np.asarray(row_data['WAVELENGTH'], dtype=float)
            flux = np.asarray(row_data['FLUX'], dtype=float)
    except Exception:
        return None

    rv = row['rv']
    if not np.isfinite(rv):
        rv = 0.0

    # 改到近似静止系
    wave_rest = wave / (1.0 + rv / c)

    mask = np.isfinite(wave_rest) & np.isfinite(flux)
    wave_rest = wave_rest[mask]
    flux = flux[mask]

    if len(wave_rest) < 20:
        return None

    order = np.argsort(wave_rest)
    wave_rest = wave_rest[order]
    flux = flux[order]

    # 去掉重复波长点
    wave_rest, unique_idx = np.unique(wave_rest, return_index=True)
    flux = flux[unique_idx]

    if len(wave_rest) < 20:
        return None

    # 必须完整覆盖目标波段
    if wave_rest.min() > common_wave[0] or wave_rest.max() < common_wave[-1]:
        return None

    flux_interp = np.interp(common_wave, wave_rest, flux)

    if not np.all(np.isfinite(flux_interp)):
        return None

    return filename, flux_interp

def load_all(star_df, common_wave, folder):
    datacube = []
    filenames = []
    valid_indices = []

    for idx, row in tqdm(star_df.iterrows(), total=len(star_df)):
        result = read_single_spectrum(row, common_wave, folder=folder)

        if result is None:
            continue

        filename, flux_interp = result
        datacube.append(flux_interp)
        filenames.append(filename)
        valid_indices.append(idx)

    datacube = np.asarray(datacube, dtype=np.float32)
    valid_indices = np.asarray(valid_indices, dtype=int)

    return datacube, filenames, valid_indices

print("正在读取全部光谱数据...")
datacube_raw, filenames_all, valid_indices = load_all(stars_df, common_wave,folder=_find('dr13_new'))

print(f"成功读入光谱数: {len(datacube_raw)}")

正在读取全部光谱数据...


  0%|          | 0/48440 [00:00<?, ?it/s]

成功读入光谱数: 37182


In [ ]:
#  归一化 + 构造最终有效样表 
from scipy import signal

def smooth_rescale_array(A, n_smooth=0, n_rescale=80, deg=3, gass_kernel=True):
    A = np.asarray(A, dtype=float)

    # 可选平滑
    if n_smooth > 0:
        window_length = 2 * n_smooth + 1
        if window_length < len(A):
            A_smooth = signal.savgol_filter(A, window_length, deg)
        else:
            A_smooth = A.copy()
    else:
        A_smooth = A.copy()

    # 连续谱估计
    if gass_kernel:
        x = np.linspace(-3, 3, 2 * n_rescale + 1)
        kernel = np.exp(-0.5 * x**2)
        kernel = kernel / kernel.sum()
    else:
        kernel = np.ones(2 * n_rescale + 1, dtype=float)
        kernel = kernel / kernel.sum()

    if len(A_smooth) > len(kernel):
        continuum = np.convolve(A_smooth, kernel, mode='same')
    else:
        continuum = np.full_like(A_smooth, np.nanmedian(A_smooth))

    # 防止除零
    continuum = np.where(np.abs(continuum) < 1e-8, np.nanmedian(continuum), continuum)

    normalized = A_smooth / continuum
    return normalized

print(f"成功加载 {len(datacube_raw)} 条光谱，正在进行归一化...")

X_list = []
for flux in tqdm(datacube_raw):
    norm_flux = smooth_rescale_array(flux, n_smooth=0, n_rescale=80, deg=3, gass_kernel=True)
    X_list.append(norm_flux)

X = np.asarray(X_list, dtype=np.float32)

# 去掉归一化失败的样本
good_mask = np.all(np.isfinite(X), axis=1)
X = X[good_mask]

stars_valid = stars_df.iloc[valid_indices].copy().reset_index(drop=True)
stars_valid['filepath'] = filenames_all
stars_valid = stars_valid.loc[good_mask].reset_index(drop=True)

print(f"光谱矩阵形状: {X.shape}")
print(f"光谱特征形状: {stars_valid.shape}")


成功加载 37182 条光谱，正在进行归一化...


  0%|          | 0/37182 [00:00<?, ?it/s]

光谱矩阵形状: (37182, 700)
光谱特征形状: (37182, 17)


## Remove duplicates and anomalies

In [ ]:
# 基于 uid 去重，保留信噪比最高的观测
print(f"去重前样本数: {len(X)}")

# 将 DataFrame 索引与光谱矩阵对应
stars_valid['__idx__'] = np.arange(len(stars_valid))

# 按 uid 分组，保留 snru 最大的行索引
keep_idx = stars_valid.loc[
    stars_valid.groupby('uid')['snru'].idxmax()
]['__idx__'].values

# 更新光谱矩阵和元数据
X = X[keep_idx]
stars_valid = stars_valid.iloc[keep_idx].reset_index(drop=True)

print(f"去重后样本数: {len(X)}")



去重前样本数: 37182
去重后样本数: 34204


In [ ]:
print(f"CN 星数: {(stars_valid['label'] == 1).sum()}")

CN 星数: 74


In [ ]:
# 基于归一化光谱的异常检测与清洗
from scipy.stats import median_abs_deviation

print(f"原始样本数: {len(X)}")

# 1. 计算每条光谱的中位数和绝对中位差 (MAD)
medians = np.median(X, axis=1)
mads = median_abs_deviation(X, axis=1)

# 2. 定义基于百分位数的阈值（剔除两端极端值）
median_low, median_high = np.percentile(medians, [0.5, 99.5])
mad_low, mad_high = np.percentile(mads, [0.5, 99.5])

# 3. 筛选正常样本
good_mask = (medians >= median_low) & (medians <= median_high) & \
            (mads >= mad_low) & (mads <= mad_high)

# 4. 应用清洗
X_clean = X[good_mask]
stars_clean = stars_valid.iloc[good_mask].reset_index(drop=True)

print(f"清洗后样本数: {len(X_clean)}")
print(f"剔除异常样本数: {len(X) - len(X_clean)}")
print(f"CN 星数: {(stars_clean['label'] == 1).sum()}")

原始样本数: 34204
清洗后样本数: 33589
剔除异常样本数: 615
CN 星数: 73
